In [21]:
from modules.explainability import ExplainableUNet3D


In [22]:
# ***Uncomment to retrain the model*** 
# import torch
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# !python modules/train.py

In [23]:
import torch
from modules.unet3d import UNet3D  # Ensure `unet3d.py` is in `modules/`

model_path = "models/brain_shift_model_fulldataset.pt"


In [24]:
import torch
from modules.explainability import ExplainableUNet3D

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Model
model = ExplainableUNet3D().to(device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()


ExplainableUNet3D(
  (encoder1): Sequential(
    (0): Conv3d(2, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    (1): BatchNorm3d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): LeakyReLU(negative_slope=0.2)
    (3): Conv3d(32, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    (4): BatchNorm3d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): LeakyReLU(negative_slope=0.2)
    (6): Dropout3d(p=0.2, inplace=False)
  )
  (encoder2): Sequential(
    (0): Conv3d(32, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    (1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): LeakyReLU(negative_slope=0.2)
    (3): Conv3d(64, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    (4): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): LeakyReLU(negative_slope=0.2)
    (6): Dropout3d(p=0.2, inplace=Fal

In [25]:
from modules.dataset import UltrasoundDataset
from torch.utils.data import DataLoader
import sys
import os

test_file = '02'
# Load dataset 
dataset = UltrasoundDataset(f"Test")
loader = DataLoader(dataset, batch_size=1, shuffle=False)


for pre, post, pre_landmarks, post_landmarks in loader:
    pre, post = pre.to(device), post.to(device)
    break


In [26]:
print("Number of samples:", len(dataset))


Number of samples: 1


In [ ]:
# Forward pass
output = model(pre, post)  # output shape: (1, 3, D, H, W)

# mean shift magnitude for scalar target
target = output.norm(p=2, dim=1).mean()

# Backward pass
model.zero_grad()
target.backward()


In [ ]:
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

# Extract gradients and activations
grads = model.get_activations_gradient()      # [1, C, D, H, W]
acts  = model.get_activations().detach()      # [1, C, D, H, W]

# Compute weighted sum (Grad-CAM) then normalize
weights = grads.mean(dim=(2, 3, 4), keepdim=True)
cam = (weights * acts).sum(dim=1).squeeze()  # [D, H, W]
cam = F.relu(cam)

cam = (cam - cam.min()) / (cam.max() - cam.min())
cam_np = cam.cpu().numpy()

# Choose a mid slice for 2D visualization
mid_slice = cam_np.shape[0] // 2

np.save(f"model_explainability/gradcam_volume_{test_file}.npy", cam_np)
print(f"Grad-CAM volume saved as gradcam_volume_{test_file}.npy")


Grad-CAM volume saved as gradcam_volume_02.npy
